# Fairness audit — experiments

MetricFrame, subgroup comparisons, threshold sweeps, ablations, and related audit tooling.

## Suggested order
Use after EDA notebooks when you have predictions or a trained pipeline to evaluate.

**Pair with:** `04_subgroup_model_students.ipynb` for subgroup training; `05_subgroup_model_working_professionals.ipynb` for the other subgroup.

---


## References

- Student notebook (Colab): https://colab.research.google.com/drive/1VNB-OmnoSawAnPf0l70a7h_fC8cNlUCu?usp=sharing

- Working professional notebook (Colab): https://colab.research.google.com/drive/1mWOM5yKHBQ6yXPmbxI0i0oY9Rwsxs5LV?usp=sharing


## 1. Environment setup

Run install cells once. Colab/Kaggle helpers are optional if data files are already local.


In [ ]:
!pip install dill

In [ ]:
# import dill
# from google.colab import drive
# drive.mount('/content/drive')

# dill.load_session('/content/drive/MyDrive/session.pkl')

In [ ]:
!pip install lime --quiet
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
!pip install catboost

In [ ]:
import io
from sklearn.model_selection import train_test_split
from google.colab import files
import pandas as pd

In [ ]:
!pip install fairlearn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, precision_score, recall_score,confusion_matrix, ConfusionMatrixDisplay
from fairlearn.metrics import (demographic_parity_difference, demographic_parity_ratio,
                               selection_rate_difference, false_negative_rate_difference,
                               false_positive_rate_difference, equalized_odds_ratio,
                               false_negative_rate, false_positive_rate)
from fairlearn.metrics import MetricFrame

In [ ]:
upload=files.upload()

In [ ]:
_train_store=pd.read_csv(io.BytesIO(upload['train.csv']),index_col='id')

In [ ]:
_train, _test = train_test_split(_train_store, test_size=0.2, random_state=42)

In [ ]:
metrics = {'accuracy': accuracy_score,
           'precision' :precision_score,
           'recall' : recall_score,
           'FNR': false_negative_rate,
           'FPR': false_positive_rate
}

## 2. Metrics — baseline (full features)


In [ ]:
upload=files.upload()

In [ ]:
y_test=_test['Depression'].reset_index(drop=True)
y_test

In [ ]:
X_test=_test.drop('Depression',axis=1)
X_test=X_test.reset_index(drop=True)
X_test

In [ ]:
upload=files.upload()

In [ ]:
y_pred_store=pd.read_csv(io.BytesIO(upload['sub_l3-weighted-ensemble_0.941338.csv']),index_col='id')

In [ ]:
y_pred=y_pred_store.copy()

In [ ]:
sensitive_feature = X_test['Gender']

calc_metric_rf = MetricFrame(metrics=metrics,
                             y_true=y_test,
                             y_pred=y_pred,
                             sensitive_features=sensitive_feature)

In [ ]:
print(f"Overall Performance:{calc_metric_rf.overall}")
calc_metric_rf.by_group

In [ ]:
plot=ConfusionMatrixDisplay(confusion_matrix(y_true=y_test,y_pred=y_pred), display_labels=["Not Depressed", "Depressed"])
plot.plot()

In [ ]:
sensitive_feature = X_test['Working Professional or Student']

calc_metric_rf = MetricFrame(metrics=metrics,
                             y_true=y_test,
                             y_pred=y_pred,
                             sensitive_features=sensitive_feature)
calc_metric_rf.by_group

In [ ]:
sensitive_feature = X_test[['Gender', 'Working Professional or Student']]

calc_metric_rf = MetricFrame(metrics=metrics,
                             y_true=y_test,
                             y_pred=y_pred,
                             sensitive_features=sensitive_feature)

calc_metric_rf.by_group

In [ ]:
from fairlearn.metrics import MetricFrame, false_negative_rate, false_positive_rate

In [ ]:
# Step 1: Combine relevant data
df = pd.DataFrame({
    'Gender': X_test['Gender'],
    'Group': X_test['Working Professional or Student'],
    'y_true': y_test,
    'y_pred': y_pred['Depression']
})

# Step 2: Store metrics
fairness_results = []

# Step 3: Loop through each group (Student and Working Professional)
for group in df['Group'].unique():
    df_group = df[df['Group'] == group]

    metric_frame = MetricFrame(
        metrics={
            'Accuracy': accuracy_score,
            'FNR': false_negative_rate,
            'FPR': false_positive_rate
        },
        y_true=df_group['y_true'],
        y_pred=df_group['y_pred'],
        sensitive_features=df_group['Gender']
    )

    df_metrics = metric_frame.by_group.reset_index()
    df_metrics['Group'] = group
    fairness_results.append(df_metrics)

# Step 4: Combine and reshape
metrics_long = pd.concat(fairness_results)
metrics_long = metrics_long.melt(id_vars=['Gender', 'Group'], var_name='Metric', value_name='Value')


In [ ]:
plt.figure(figsize=(10, 6))
g = sns.catplot(
    data=metrics_long,
    kind='bar',
    x='Metric', y='Value',
    hue='Gender',
    col='Group',
    palette='Set2',
    height=5,
    aspect=1
)

g.set_titles("{col_name}")
g.set_axis_labels("Metric", "Score")
g.add_legend(title="Gender")
plt.suptitle("Accuracy, FNR, and FPR by Gender and Group", y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
df = X_test.copy()

# Step 1: Define city to region
city_to_region = {
    'Ludhiana': 'North', 'Varanasi': 'North', 'Kanpur': 'North', 'Lucknow': 'North',
    'Meerut': 'North', 'Agra': 'North', 'Ghaziabad': 'North',
    'Visakhapatnam': 'Southeast', 'Krishna': 'Southeast',
    'Mumbai': 'Southwest', 'Thane': 'Southwest', 'Nashik': 'Southwest', 'Pune': 'Southwest',
    'Nagpur': 'Southwest', 'Kalyan': 'Southwest', 'Vasai-Virar': 'Southwest',
    'Ahmedabad': 'Centralwest', 'Rajkot': 'Centralwest', 'Surat': 'Centralwest', 'Vadodara': 'Centralwest',
    'Bangalore': 'Southwest',
    'Patna': 'Centraleast',
    'Jaipur': 'Northwest',
    'Faridabad': 'North', 'Gurgaon': 'North',
    'Hyderabad': 'Southeast',
    'Srinagar': 'North',
    'Kolkata': 'Centraleast',
    'Chennai': 'Southeast',
    'Bhopal': 'Centralwest', 'Indore': 'Centralwest', 'Morena': 'Centralwest',
    'Delhi': 'North'
}

# Step 2: Map city to region
df['Region'] = df['City'].map(city_to_region).fillna('Other')

# Step 3: Map city to state (needed for belt mapping)
city_to_state = {
    'Ludhiana': 'Punjab', 'Varanasi': 'Uttar Pradesh', 'Kanpur': 'Uttar Pradesh',
    'Lucknow': 'Uttar Pradesh', 'Meerut': 'Uttar Pradesh', 'Agra': 'Uttar Pradesh',
    'Ghaziabad': 'Uttar Pradesh', 'Visakhapatnam': 'Andhra Pradesh', 'Krishna': 'Andhra Pradesh',
    'Mumbai': 'Maharashtra', 'Thane': 'Maharashtra', 'Nashik': 'Maharashtra', 'Pune': 'Maharashtra',
    'Nagpur': 'Maharashtra', 'Kalyan': 'Maharashtra', 'Vasai-Virar': 'Maharashtra',
    'Ahmedabad': 'Gujarat', 'Rajkot': 'Gujarat', 'Surat': 'Gujarat', 'Vadodara': 'Gujarat',
    'Bangalore': 'Karnataka', 'Patna': 'Bihar', 'Jaipur': 'Rajasthan',
    'Faridabad': 'Haryana', 'Gurgaon': 'Haryana', 'Hyderabad': 'Telangana',
    'Srinagar': 'Jammu and Kashmir', 'Kolkata': 'West Bengal', 'Chennai': 'Tamil Nadu',
    'Bhopal': 'Madhya Pradesh', 'Indore': 'Madhya Pradesh', 'Morena': 'Madhya Pradesh',
    'Delhi': 'Delhi'
}
df['State'] = df['City'].map(city_to_state)

# Step 4: Define belt mapping
state_to_belt = {
    'Maharashtra': 'Lower Dominated Belt (CW)', 'Gujarat': 'Lower Dominated Belt (CW)',
    'Delhi': 'Upper Dominated Belt (CN)', 'Uttar Pradesh': 'Upper Dominated Belt (CN)',
    'Punjab': 'Upper Dominated Belt (CN)', 'Haryana': 'Upper Dominated Belt (CN)',
    'Bihar': 'Upper Dominated Belt (CN)',
    'Madhya Pradesh': 'Other', 'West Bengal': 'Other', 'Andhra Pradesh': 'Other',
    'Jammu and Kashmir': 'Other', 'Telangana': 'Other', 'Rajasthan': 'Other',
    'Karnataka': 'Other', 'Tamil Nadu': 'Other'
}
df['Belt'] = df['State'].map(state_to_belt)

# Step 5: Attach ground truth and predictions
df['y_true'] = y_test.values
df['y_pred'] = y_pred.values

# Step 7: Compute metrics per belt
belt_metrics = []
for belt, group in df.groupby('Belt'):
    acc = accuracy_score(group['y_true'], group['y_pred'])
    fnr = false_negative_rate(group['y_true'], group['y_pred'])
    fpr = false_positive_rate(group['y_true'], group['y_pred'])
    belt_metrics.append({'Belt': belt, 'Accuracy': acc, 'FNR': fnr, 'FPR': fpr})

belt_df = pd.DataFrame(belt_metrics).set_index('Belt')

# Step 8: Plot
belt_df.plot(kind='bar', figsize=(6, 4))
plt.title('Model Performance by Belt')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
df['Belt'] = df['Belt'].fillna('Unknown').astype(str)

sensitive_feature = df[['Gender','Belt']]

calc_metric_rf = MetricFrame(metrics=metrics,
                             y_true=y_test,
                             y_pred=y_pred,
                             sensitive_features=sensitive_feature)

calc_metric_rf.by_group

In [ ]:
# Ensure clean types
# df['Gender'] = df['Gender'].fillna('Unknown').astype(str)
# df['Belt'] = df['Belt'].fillna('Unknown').astype(str)

# List to collect results
belt_parity_results = []

# Iterate over each belt group
for belt_name, group in df.groupby('Belt'):
    fnr_diff = false_negative_rate_difference(group['y_true'], group['y_pred'], sensitive_features=group['Gender'])
    fpr_diff = false_positive_rate_difference(group['y_true'], group['y_pred'], sensitive_features=group['Gender'])
    sel_diff = selection_rate_difference(group['y_true'], group['y_pred'], sensitive_features=group['Gender'])

    belt_parity_results.append({
        'Belt': belt_name,
        'FNR Difference': fnr_diff,
        'FPR Difference': fpr_diff,
        'Selection Rate Difference': sel_diff
    })

# Create a DataFrame
belt_parity_df = pd.DataFrame(belt_parity_results).set_index('Belt')
# Remove 'Unknown' belt if present
belt_parity_df = belt_parity_df[belt_parity_df.index != 'Unknown']


# Plot the disparities
ax = belt_parity_df.plot(kind='bar', figsize=(10, 6))
plt.title('Gender Parity Metrics by Belt')
plt.ylabel('Disparity (Max Difference)')
plt.xticks(rotation=45)
plt.tight_layout()

plt.title('Gender Parity Metrics by Belt')
plt.ylabel('Disparity (Max Difference)')
plt.xticks(rotation=0)
plt.tight_layout()

# Annotate each bar with percentage text
for p in ax.patches:
    height = p.get_height()
    if not pd.isna(height):
        ax.annotate(f'{height:.1%}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=10, xytext=(0, 2),
                    textcoords='offset points')

plt.show()


plt.show()


## 3. Metrics — after dropping Name and City


In [ ]:
y_test_drop=_test['Depression'].reset_index(drop=True)
y_test_drop

In [ ]:
X_test_drop=_test.drop('Depression',axis=1)
X_test_drop=X_test_drop.reset_index(drop=True)
X_test_drop

In [ ]:
upload=files.upload()

In [ ]:
y_pred_drop_store=pd.read_csv(io.BytesIO(upload['sub_l3-weighted-ensemble_0.940316.csv']),index_col='id')

In [ ]:
y_pred_drop=y_pred_drop_store.copy()

In [ ]:
y_pred_drop

In [ ]:
sensitive_feature = X_test_drop['Gender']

calc_metric_rf = MetricFrame(metrics=metrics,
                             y_true=y_test_drop,
                             y_pred=y_pred_drop,
                             sensitive_features=sensitive_feature)

In [ ]:
print(f"Overall Performance:{calc_metric_rf.overall}")
calc_metric_rf.by_group

In [ ]:
plot=ConfusionMatrixDisplay(confusion_matrix(y_true=y_test_drop,y_pred=y_pred_drop), display_labels=["Not Depressed", "Depressed"])
plot.plot()

In [ ]:
# Compute metric values for both predictions
results = {
    'Original Prediction': {metric: func(y_test, y_pred['Depression']) for metric, func in metrics.items()},
    'After Dropping Name & City': {metric: func(y_test, y_pred_drop['Depression']) for metric, func in metrics.items()}
}

# Convert to DataFrame
df_metrics = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Version'})

# Melt for plotting
df_melted = df_metrics.melt(id_vars='Version', var_name='Metric', value_name='Score')

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=df_melted, x='Metric', y='Score', hue='Version')
plt.ylim(0, 1)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("")
plt.xticks(rotation=15)
plt.legend(title="Model Version")
plt.tight_layout()
plt.show()

## 4. Metrics — student subgroup


In [ ]:
_train_student=_train[_train['Working Professional or Student']=='Student']
y_train_student=_train_student['Depression'].reset_index(drop=True)
X_train_student=_train_student.drop('Depression',axis=1).reset_index(drop=True)
X_train_student=X_train_student.drop(['Working Professional or Student','Profession','Work Pressure','Job Satisfaction',], axis=1)
X_train_student.info()

In [ ]:
_test_student=_test[_test['Working Professional or Student']=='Student']
# _test_student

In [ ]:
y_test_student=_test_student['Depression'].reset_index(drop=True)
y_test_student

In [ ]:
X_test_student=_test_student.drop('Depression',axis=1).reset_index(drop=True)
X_test_student=X_test_student.drop(['Working Professional or Student','Profession','Work Pressure','Job Satisfaction',], axis=1)

X_test_student

In [ ]:
palette = {0: sns.color_palette("Set1")[1],  # Blue-ish
           1: sns.color_palette("Set1")[0]}  # Red-ish

In [ ]:
# Define the palette
palette = {0: sns.color_palette("Set1")[1],  # Not Depressed - Blue
           1: sns.color_palette("Set1")[0]}  # Depressed - Red

# Class label mapping
class_mapping = {0: 'Not Depressed', 1: 'Depressed'}

# Compute proportions
label_percentages = y_train_student.value_counts(normalize=True).sort_index()
df = pd.DataFrame({'Label': label_percentages.index, 'Proportion': label_percentages.values})
df['Color'] = df['Label'].map(palette)

# Plot
plt.figure(figsize=(6, 4))
bars = plt.bar(df['Label'], df['Proportion'], color=df['Color'])
#
plt.ylabel('Proportion')
plt.xlabel('')  # No x-axis label
plt.ylim(0, 1.05)
# plt.yticks([])
plt.xticks([])  # No tick labels on x-axis

# Annotate bars
for row in df.itertuples():
    plt.text(row.Label, row.Proportion + 0.02,
             f'{class_mapping[row.Label]} ({row.Proportion:.2%})',
             ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()



In [ ]:
upload=files.upload()

In [ ]:
y_pred_student_store=pd.read_csv(io.BytesIO(upload['sub_l3-weighted-ensemble_0.852107.csv']),index_col='id')

In [ ]:
y_pred_student=y_pred_student_store
y_pred_student['Depression'].value_counts()

In [ ]:
# Count values
pred_counts = y_pred_student['Depression'].value_counts().sort_index()
true_counts = y_test_student.value_counts().sort_index()

# Prepare data for seaborn
plot_data = pd.DataFrame({
    'Class': ['Not Depressed (0)', 'Depressed (1)'] * 2,
    'Count': list(pred_counts) + list(true_counts),
    'Type': ['Predicted'] * 2 + ['Actual'] * 2
})

# Plot
plt.figure(figsize=(6, 4))
sns.barplot(data=plot_data, x='Class', y='Count', hue='Type')
plt.title('Depression Counts: Predicted vs Actual')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Combine actual data
df = pd.DataFrame({
    'Gender': X_test_student['Gender'],
    'Predicted': y_pred_student['Depression'],
    'Actual': y_test_student
})

# Separate male and female subsets
df_male = df[df['Gender'] == 'Male']
df_female = df[df['Gender'] == 'Female']

# Count for males
male_counts = pd.DataFrame({
    'Type': ['Predicted', 'Actual'],
    'Not Depressed': [(df_male['Predicted'] == 0).sum(), (df_male['Actual'] == 0).sum()],
    'Depressed': [(df_male['Predicted'] == 1).sum(), (df_male['Actual'] == 1).sum()]
})
male_melted = male_counts.melt(id_vars='Type', var_name='Depression', value_name='Count')

# Count for females
female_counts = pd.DataFrame({
    'Type': ['Predicted', 'Actual'],
    'Not Depressed': [(df_female['Predicted'] == 0).sum(), (df_female['Actual'] == 0).sum()],
    'Depressed': [(df_female['Predicted'] == 1).sum(), (df_female['Actual'] == 1).sum()]
})
female_melted = female_counts.melt(id_vars='Type', var_name='Depression', value_name='Count')

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

sns.barplot(data=male_melted, x='Depression', y='Count', hue='Type', ax=axes[0])
axes[0].set_title('Male Students')
axes[0].set_xlabel('Depression Status')

sns.barplot(data=female_melted, x='Depression', y='Count', hue='Type', ax=axes[1])
axes[1].set_title('Female Students')
axes[1].set_xlabel('Depression Status')

for ax in axes:
    ax.set_ylabel('Number of Students')
    ax.legend(title='Type')

plt.suptitle('Predicted vs Actual Depression Counts by Gender')
plt.tight_layout()
plt.show()


In [ ]:
sensitive_feature = X_test_student['Gender']

calc_metric_rf = MetricFrame(metrics=metrics,
                             y_true=y_test_student,
                             y_pred=y_pred_student,
                             sensitive_features=sensitive_feature)

In [ ]:
print(f"Overall Performance:{calc_metric_rf.overall}")
calc_metric_rf.by_group

In [ ]:
plot=ConfusionMatrixDisplay(confusion_matrix(y_true=y_test_student,y_pred=y_pred_student), display_labels=["Not Depressed", "Depressed"])
plot.plot()

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)
accuracies = []
fnrs = []
fprs = []

for thr in thresholds:
    y_pred_binary = (student_preds >= thr).astype(int)
    acc = accuracy_score(y_test_student, y_pred_binary)
    accuracies.append(acc)

    tn, fp, fn, tp = confusion_matrix(y_test_student, y_pred_binary).ravel()

    fnr = fn / (fn + tp) if (fn + tp) != 0 else 0  # False Negatives / Actual Positives
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0  # False Positives / Actual Negatives

    fnrs.append(fnr)
    fprs.append(fpr)


plt.figure(figsize=(10, 6))
plt.plot(thresholds, accuracies, marker='o', label='Accuracy')
plt.plot(thresholds, fnrs, marker='s', label='False Negative Rate (FNR)')
plt.plot(thresholds, fprs, marker='^', label='False Positive Rate (FPR)')

plt.title('Model Metrics vs Threshold')
plt.xlabel('Threshold')
plt.ylabel('Metric Value')
plt.legend()
plt.grid(True)
plt.xticks(thresholds)
plt.show()


In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Define your mappings
X_train_student_modified=X_train_student.copy()
X_test_student_modified=X_test_student.copy()
mapping_rules = {
    'Dietary Habits': {'Unhealthy': 2, 'Moderate': 1, 'Healthy': 0},
    'Sleep Duration': {'Less than 5 hours': 3, '5-6 hours': 2, 'More than 8 hours': 1, '7-8 hours': 0},
    'Have you ever had suicidal thoughts ?': {'Yes': 1, 'No': 0},
    'Gender': {'Male': 0, 'Female': 1},
    'Family History of Mental Illness':{'Yes': 1, 'No': 0},
    'Degree' : {'Class 12': 0,   'BA': 1, 'B.Com': 1, 'B.Sc': 1, 'BSc': 1, 'B.Tech': 1, 'BCA': 1, 'BBA': 1,
    'B.Arch': 1, 'B.Ed': 1, 'BHM': 1, 'B.Pharm': 1, 'LLB': 1, 'BE': 1, 'BH': 1, 'B': 1,
    'MA': 2, 'MBA': 2, 'M.Com': 2, 'M.Tech': 2, 'MCA': 2, 'M.Ed': 2,
    'MSc': 2, 'M.Pharm': 2, 'ME': 2, 'LLM': 2, 'MPA': 2, 'MHM': 2,
    'PhD': 3,
    }
}

# Apply mappings, fallback to -1 for unexpected values
for col, mapping in mapping_rules.items():
    if col in X_test_student_modified.columns:
        X_test_student_modified[col] = X_test_student_modified[col].map(mapping).fillna(-1).astype(int)
        X_train_student_modified[col]=X_train_student_modified[col].map(mapping).fillna(-1).astype(int)


# Encode 'Name' and 'City' if present
for col in ['Name', 'City']:
    if col in X_test_student_modified.columns:
        le = LabelEncoder()

        # Fit on combined data to keep consistency
        combined = pd.concat([X_train_student_modified[col], X_test_student_modified[col]], axis=0)
        le.fit(combined)

        X_train_student_modified[col] = le.transform(X_train_student_modified[col])
        X_test_student_modified[col] = le.transform(X_test_student_modified[col])



In [ ]:
X_test_student_modified['Degree'].unique()

In [ ]:
upload=files.upload()

In [ ]:
student_preds=pd.read_csv(io.BytesIO(upload['weighted_test_preds_stu.csv']))

In [ ]:
student_preds = student_preds.iloc[:, 0]

In [ ]:
n, bins, patches = plt.hist(student_preds, bins=6, range=[-0.25, 1.25], edgecolor='black')
# patches[0].set_facecolor('red')
plt.title("Distribution of Output Probabilites for Students")
plt.xlabel('Weighted Probabilites')
plt.ylabel('Count')

### 6.1 SHAP (surrogate)


Using CatboostClassifier as Surrogate for the original model

In [ ]:
from catboost import CatBoostClassifier

surrogate_cb = CatBoostClassifier(
    iterations=1,
    depth=3,
    learning_rate=0.1,
    verbose=0,
    random_state=42
)

surrogate_cb.fit(X_test_student_modified, y_pred_student)

In [ ]:
fidelity = accuracy_score(y_pred_student, surrogate_cb.predict(X_test_student_modified))
print(f"Fidelity to ensemble: {fidelity:.4f}")

In [ ]:
feature_importances = surrogate_cb.get_feature_importance()
sns.barplot(x=feature_importances, y=X_test_student_modified.columns)
plt.title('CatBoost Surrogate Feature Importances')
plt.show()

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(surrogate_cb)
shap_values = explainer.shap_values(X_test_student_modified)

shap.summary_plot(shap_values, X_test_student_modified,)

In [ ]:
idx=0
print("Correct:", "Depressed" if y_test_student[idx] else "Not Depressed")
print("Classified:", "Depressed" if y_pred_student['Depression'][idx] else "Not Depressed")
shap.bar_plot(shap_values[1], feature_names=X_test_student_modified.columns )

In [ ]:
fpr=[]
j=0
for i in range(y_pred_student.size):
  # if(j==10):break
  if(y_pred_student['Depression'][i]==1 and y_test_student[i]==0):
    fpr.append(i)
    j+=1
# fpr
j

In [ ]:
for idx in fpr:
  print("Correct:", "Depressed" if y_test_student[idx] else "Not Depressed")
  print("Classified:", "Depressed" if y_pred_student['Depression'][idx] else "Not Depressed")
  shap.bar_plot(shap_values[idx], feature_names=X_test_student_modified.columns )
  for j in X_test_student.columns:
    shap_values['']

In [ ]:
X_test_student_fpr=pd.DataFrame(X_test_student_modified,index=fpr)
shap_values_fpr=shap_values[fpr]
shap.summary_plot(shap_values_fpr, X_test_student_fpr)

### 6.2 LIME


In [ ]:
explainer = LimeTabularExplainer(
    training_data=X_train_student_modified.values,
    feature_names=X_test_student_modified.columns.tolist(),
    class_names=['Not Depressed', 'Depressed'],
    mode='classification',
    discretize_continuous=False
)

In [ ]:
i =5  # index of the student to explain

exp = explainer.explain_instance(
    data_row=X_test_student_modified.iloc[i].values,
    predict_fn=surrogate_cb.predict_proba,
    num_features=10
)

# Step 4: Show explanation
exp.show_in_notebook(show_table=True)

Lime explanations for 10 FP Classifications


In [ ]:
for idx in fpr[:5]:
    prob = student_preds[idx]
    predict_fn = lambda x, p=prob: np.tile([1 - p, p], (x.shape[0], 1))
    exp = explainer.explain_instance(
        data_row=X_test_student_modified.iloc[idx].values,
        predict_fn=surrogate_cb.predict_proba,
        num_features=10
    )
    exp.show_in_notebook(show_table=True)

Top 5 points for Explanations

In [ ]:
from lime import submodular_pick

sp_obj = submodular_pick.SubmodularPick(
    explainer=explainer,
    data=X_test_student_modified.values,
    predict_fn=surrogate_cb.predict_proba,
    sample_size=10,  # use full data
    num_features=10,
    num_exps_desired=5
)

for i, idx in enumerate(sp_obj.V):  # sp_obj.V gives selected row indices
    exp = explainer.explain_instance(
        data_row=X_test_student_modified.iloc[idx].values,
        predict_fn=surrogate_cb.predict_proba,
        num_features=10
    )

    actual = y_test_student.iloc[idx] if isinstance(y_test_student, pd.Series) else y_test_student[idx]
    predicted_prob = student_preds[idx]
    predicted_class = int(predicted_prob > 0.5)

    print(f"🔹 Explanation {i + 1}")
    print(f"Index: {idx}")
    print(f"Predicted Probability: {predicted_prob:.4f}")
    print(f"Predicted Class: {predicted_class}")
    print(f"Actual Class: {actual}")
    exp.show_in_notebook(show_table=True)
    print("=" * 40)



### 6.3 Plots


In [ ]:
counts = y_pred.loc[X_test['Working Professional or Student'] == 'Student', 'Depression'].reset_index(drop=True)
# counts.value_counts()
counts2=y_pred_student['Depression'].reset_index(drop=True)
# counts2.info()

index=[0,1]
data=pd.DataFrame({'True Value':y_test_student.value_counts().reindex(index),
                   'Original Model':counts.value_counts().reindex(index),
                   'Student Model':counts2.value_counts().reindex(index)}).fillna(0).astype(int)
ax=data.plot(kind='bar',title="Distribution of Depression in Students")
ax.set_xlabel("Label (0 = No Depression, 1 = Depression)")
ax.set_ylabel("Count")
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Ground truth
true_vals = y_test_student.reset_index(drop=True)

# Model predictions
counts = y_pred.loc[X_test['Working Professional or Student'] == 'Student', 'Depression'].reset_index(drop=True)
counts2 = y_pred_student['Depression'].reset_index(drop=True)

# Define metric functions
metrics = {
    'Accuracy': accuracy_score,
    'Precision': precision_score,
    'Recall': recall_score,
    'FNR': false_negative_rate,
    'FPR': false_positive_rate
}

# Evaluate both models
results = {'Metric': [], 'Model': [], 'Value': []}

for name, func in metrics.items():
    results['Metric'].append(name)
    results['Model'].append('Original Model')
    results['Value'].append(func(true_vals, counts))

    results['Metric'].append(name)
    results['Model'].append('Student Model')
    results['Value'].append(func(true_vals, counts2))

# Create DataFrame
results_df = pd.DataFrame(results)

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=results_df, x='Metric', y='Value', hue='Model', palette='Set2')
ax.set_title("Evaluation Metrics for Student Dataset")
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", label_type="edge", padding=2)
plt.tight_layout()
plt.grid(axis='y')
plt.show()


## 5. Metrics — working professional subgroup


In [ ]:
_train_prof=_train[_train['Working Professional or Student']!='Student']
y_train_prof=_train_prof['Depression'].reset_index(drop=True)
# X_train_student=_train_student.drop('Depression',axis=1).reset_index(drop=True)
# X_train_student=X_train_student.drop(['Working Professional or Student','Profession','Work Pressure','Job Satisfaction',], axis=1)
# X_train_student.info()

In [ ]:
upload=files.upload()

In [ ]:
prof_preds=pd.read_csv(io.BytesIO(upload['weighted_test_preds_prof.csv']))

In [ ]:
prof_preds=prof_preds.iloc[:,0]

In [ ]:
prof_preds.info()

In [ ]:
n, bins, patches = plt.hist(prof_preds, bins=6, range=[-0.25, 1.25], edgecolor='black')
patches[0].set_facecolor('red')
plt.title("Distribution of Output Probabilites for Working Professionals")
plt.xlabel('Weighted Probabilites')
plt.ylabel('Count')

In [ ]:
_test_prof=_test[_test['Working Professional or Student']!='Student']
_test_prof.info()

In [ ]:
y_test_prof=_test_prof['Depression'].reset_index(drop=True)
y_test_prof

In [ ]:
X_test_prof=_test_prof.drop('Depression',axis=1).reset_index(drop=True)
X_test_prof=X_test_prof.drop(['Working Professional or Student','Academic Pressure','CGPA','Study Satisfaction'], axis=1)
X_test_prof.info()

In [ ]:
upload=files.upload()

In [ ]:
y_pred_prof_store=pd.read_csv(io.BytesIO(upload['sub_l3-weighted-ensemble_0.963029.csv']),index_col='id')

In [ ]:
y_pred_prof=y_pred_prof_store.copy()

In [ ]:
y_pred_prof.info()

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)
accuracies = []
fnrs = []
fprs = []

for thr in thresholds:
    y_pred_binary = (prof_preds >= thr).astype(int)
    acc = accuracy_score(y_test_prof, y_pred_binary)
    accuracies.append(acc)

    tn, fp, fn, tp = confusion_matrix(y_test_prof, y_pred_binary).ravel()

    fnr = fn / (fn + tp) if (fn + tp) != 0 else 0  # False Negatives / Actual Positives
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0  # False Positives / Actual Negatives

    fnrs.append(fnr)
    fprs.append(fpr)


plt.figure(figsize=(10, 6))
plt.plot(thresholds, accuracies, marker='o', label='Accuracy')
plt.plot(thresholds, fnrs, marker='s', label='False Negative Rate (FNR)')
plt.plot(thresholds, fprs, marker='^', label='False Positive Rate (FPR)')

plt.title('Model Metrics vs Threshold')
plt.xlabel('Threshold')
plt.ylabel('Metric Value')
plt.legend()
plt.grid(True)
plt.xticks(thresholds)
plt.show()


In [ ]:
# Define the palette
palette = {0: sns.color_palette("Set1")[1],  # Not Depressed - Blue
           1: sns.color_palette("Set1")[0]}  # Depressed - Red

# Class label mapping
class_mapping = {0: 'Not Depressed', 1: 'Depressed'}

# Compute proportions
label_percentages = y_train_prof.value_counts(normalize=True).sort_index()
df = pd.DataFrame({'Label': label_percentages.index, 'Proportion': label_percentages.values})
df['Color'] = df['Label'].map(palette)

# Plot
plt.figure(figsize=(6, 4))
bars = plt.bar(df['Label'], df['Proportion'], color=df['Color'])
#
plt.ylabel('Proportion')
plt.xlabel('')  # No x-axis label
plt.ylim(0, 1.05)
# plt.yticks([])
plt.xticks([])  # No tick labels on x-axis

# Annotate bars
for row in df.itertuples():
    plt.text(row.Label, row.Proportion + 0.02,
             f'{class_mapping[row.Label]} ({row.Proportion:.2%})',
             ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()



In [ ]:
sensitive_feature=X_test_prof['Gender']

calc_metric_rf=MetricFrame(metrics=metrics,y_true=y_test_prof,y_pred=y_pred_prof,sensitive_features=sensitive_feature)

In [ ]:
print(f"Overall Performance:{calc_metric_rf.overall}")
calc_metric_rf.by_group

In [ ]:
plot=ConfusionMatrixDisplay(confusion_matrix(y_true=y_test_prof,y_pred=y_pred_prof),display_labels=['Not Depressed','Depressed'])
plot.plot()

In [ ]:
# Ground truth
true_vals = y_test_prof.reset_index(drop=True)

# Model predictions
counts = y_pred.loc[X_test['Working Professional or Student'] != 'Student', 'Depression'].reset_index(drop=True)
counts2 = y_pred_prof['Depression'].reset_index(drop=True)

# Define metric functions
metrics = {
    'Accuracy': accuracy_score,
    'Precision': precision_score,
    'Recall': recall_score,
    'FNR': false_negative_rate,
    'FPR': false_positive_rate
}

# Evaluate both models
results = {'Metric': [], 'Model': [], 'Value': []}

for name, func in metrics.items():
    results['Metric'].append(name)
    results['Model'].append('Original Model')
    results['Value'].append(func(true_vals, counts))

    results['Metric'].append(name)
    results['Model'].append('Professionals Model')
    results['Value'].append(func(true_vals, counts2))

# Create DataFrame
results_df = pd.DataFrame(results)

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=results_df, x='Metric', y='Value', hue='Model', palette='Set2')
ax.set_title("Evaluation Metrics for Working Professionals Dataset")
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", label_type="edge", padding=2)
plt.tight_layout()
plt.grid(axis='y')
plt.show()


In [ ]:
# Inputs
thresholds = np.arange(0.1, 1.0, 0.1)

# Containers
metrics = {
    'accuracy': {'student': [], 'prof': []},
    'fnr': {'student': [], 'prof': []},
    'fpr': {'student': [], 'prof': []}
}

def compute_metrics(y_true, y_probs):
    accs, fnrs, fprs = [], [], []
    for thr in thresholds:
        y_pred = (y_probs >= thr).astype(int)
        accs.append(accuracy_score(y_true, y_pred))

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        fnrs.append(fnr)
        fprs.append(fpr)
    return accs, fnrs, fprs

# Compute for students
acc_s, fnr_s, fpr_s = compute_metrics(y_test_student, student_preds)
metrics['accuracy']['student'] = acc_s
metrics['fnr']['student'] = fnr_s
metrics['fpr']['student'] = fpr_s

# Compute for professionals
acc_p, fnr_p, fpr_p = compute_metrics(y_test_prof, prof_preds)
metrics['accuracy']['prof'] = acc_p
metrics['fnr']['prof'] = fnr_p
metrics['fpr']['prof'] = fpr_p

# Plotting
plt.figure(figsize=(12, 7))

# Accuracy
plt.plot(thresholds, metrics['accuracy']['prof'], marker='P', linestyle='--', label='Accuracy - Professional',color='blue')
plt.plot(thresholds, metrics['accuracy']['student'], marker='o', label='Accuracy - Student', color='blue')

# FNR
plt.plot(thresholds, metrics['fnr']['prof'], marker='P', linestyle='--', label='FNR - Professional',color='red')
plt.plot(thresholds, metrics['fnr']['student'], marker='o', label='FNR - Student',color='red')

# FPR
plt.plot(thresholds, metrics['fpr']['prof'], marker='P', linestyle='--', label='FPR - Professional',color='green')
plt.plot(thresholds, metrics['fpr']['student'], marker='o', label='FPR - Student',color='green')

plt.title('Metrics vs Threshold (Students vs Professionals)')
plt.xlabel('Threshold')
plt.ylabel('Metric Value')
plt.legend()
plt.grid(True)
plt.xticks(thresholds)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


In [ ]:
import dill
from google.colab import drive
drive.mount('/content/drive')

with open('/content/drive/MyDrive/session.pkl', 'wb') as f:
    dill.dump_session(f)